<a href="https://colab.research.google.com/github/fataei/Deep-Learning-Projects/blob/main/Resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import nibabel as nib
import glob
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as TF
from torch.utils.data import Dataset
from torchvision import transforms
import pandas as pd
#dir(models)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

def extract_folder(zip_file_path, destination_folder):
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            # Get a list of all files and folders in the ZIP archive
            all_files = zip_ref.namelist()

            # Filter files and folders to include only those in the specified folder
            folder_files = [file for file in all_files]

            # Extract files and folders to the destination folder
            for file in folder_files:
                zip_ref.extract(file, destination_folder)

        print(f"Successfully extracted the  folder.")
    except zipfile.BadZipFile:
        print("Error: The file is not a valid ZIP archive.")
    except FileNotFoundError:
        print("Error: File not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
extract_folder("/content/drive/MyDrive/Br35H.zip" ,".")


Successfully extracted the  folder.


In [ ]:
drive.flush_and_unmount()

In [ ]:
# model=torchvision.models.resnet50(pretrained=True)
# print(model)
from torchvision import models
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 129MB/s]


In [ ]:
# model.conv1
# model.fc
# model.layer1[0].conv1.weight.shape

In [ ]:
import torch.nn as nn
#print(model.fc)

model.fc=nn.Linear(2048,1024)
model.classifier = nn.Sequential(
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(1024, 512),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(512, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(256,2),
    nn.ReLU(inplace=True)
)
print(model.classifier)

Sequential(
  (0): ReLU(inplace=True)
  (1): Dropout(p=0.5, inplace=False)
  (2): Linear(in_features=1024, out_features=512, bias=True)
  (3): ReLU(inplace=True)
  (4): Dropout(p=0.5, inplace=False)
  (5): Linear(in_features=512, out_features=256, bias=True)
  (6): ReLU(inplace=True)
  (7): Dropout(p=0.5, inplace=False)
  (8): Linear(in_features=256, out_features=2, bias=True)
  (9): ReLU(inplace=True)
)


In [ ]:
# nn.Sequential(*list(model.children())[:-2])

In [ ]:
# model.requires_grad_(True)
# # model.fc.requires_grad_(True)
# # model.layer4.requires_grad_(True)

In [ ]:
for name, child in model.named_children():
   if name in ['layer3', 'layer4','fc',"classifier"]:
       print(name + ' is unfrozen')
       for param in child.parameters():
           param.requires_grad = True
   else:
       print(name + ' is frozen')
       for param in child.parameters():
           param.requires_grad = False

conv1 is frozen
bn1 is frozen
relu is frozen
maxpool is frozen
layer1 is frozen
layer2 is frozen
layer3 is unfrozen
layer4 is unfrozen
avgpool is frozen
fc is unfrozen
classifier is unfrozen


In [ ]:
def num_trainable_params(model):
  nums = sum(p.numel() for p in model.parameters())
  return nums

In [ ]:
num_trainable_params(model)

26262850

In [ ]:
#imports
import os
import nibabel as nib
import glob
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as TF
from torch.utils.data import Dataset
from torchvision import transforms
import pandas as pd


In [ ]:
import sys
from pathlib import Path
try:
  os.mkdir("./converted_dataset")
  os.mkdir("./converted_dataset/yes")
  os.mkdir("./converted_dataset/no")
except:
  pass
# Reading labels
tumorous = os.listdir("yes")
non_tumorous = os.listdir("no")


import numpy as np
from PIL import Image
def to_np(image_dir):
  #image_dir = '/content/train/cityscapes_data/train'
  files = os.listdir(image_dir)
  # train_data = []
  # train_labels = []
  for image in files:
    image_1 = Image.open(os.path.join(image_dir , image))
    image_2 = np.asarray(image_1)
    #label=RGBtoOneHot(label,color2label)
    np.save("/content/converted_dataset/"+image_dir+'/'+image,image_2)

to_np("yes")
to_np("no")





# NoHemorrhage = hemorrhage_diagnosis_array[:, 7]
# np.set_printoptions(threshold=sys.maxsize)
# np.save(os.path("./converted_dataset/label"),NoHemorrhage)




In [ ]:
a=os.listdir("./converted_dataset/yes")
num_yes = len(a)
num_no=len(os.listdir("./converted_dataset/no"))
print(num_yes)

1500


In [ ]:
path1 = "./converted_dataset/train_img/"
path2 = "./converted_dataset/test_img/"
path3 = "./converted_dataset/test_label/"
path4 = "./converted_dataset/train_label/"
try:
  os.mkdir(path1[:-1])
except:
  for i in os.listdir(path1):
      os.remove(path1+i)
try:
  os.mkdir(path2[:-1])
except:
  for i in os.listdir(path2):
      os.remove(path2+i)
try:
  os.mkdir(path3[:-1])
except:
  for i in os.listdir(path3):
      os.remove(path3+i)
try:
  os.mkdir(path4[:-1])
except:

  for i in os.listdir(path4):
      os.remove(path4+i)



In [ ]:
import random

arr=[]
val_h=[]
yes_files=os.listdir("./converted_dataset/yes")
no_files=os.listdir("./converted_dataset/no")
while len(arr)<(num_yes)*(20/100):
    a=random.randint(0,num_yes-1)
    if a not in arr:
        val_h.append(yes_files[a])
        arr.append(a)
        ct_name=np.load("./converted_dataset/yes/"+str(yes_files[a]))
        np.save("./converted_dataset/test_img/" + str(yes_files[a]).split(".")[0],ct_name.astype(np.uint8))
        np.save("./converted_dataset/test_label/" + str(yes_files[a]).split(".")[0],[1])

val_nh=[]
arr2=[]
while len(arr2)<num_no*(20/100):
    a=random.randint(0,num_no-1)
    if a not in arr2:
        val_nh.append(no_files[a])
        arr2.append(a)
        ct_name=np.load("./converted_dataset/no/"+str(no_files[a]))
        np.save("./converted_dataset/test_img/" + str(no_files[a]).split(".")[0],ct_name.astype(np.uint8))
        np.save("./converted_dataset/test_label/" + str(no_files[a]).split(".")[0],[0])

for x in yes_files:
    if x not in val_h+val_nh:
        ct_name=np.load("./converted_dataset/yes/"+str(x))
        np.save("./converted_dataset/train_img/" + str(x),ct_name.astype(np.uint8))
        np.save("./converted_dataset/train_label/" + str(x).split(".")[0],[1])
for x in no_files:
    if x not in val_h+val_nh:
        ct_name=np.load("./converted_dataset/no/"+str(x))
        np.save("./converted_dataset/train_img/" + str(x),ct_name.astype(np.uint8))
        np.save("./converted_dataset/train_label/" + str(x),[0])



In [ ]:
len(os.listdir("./converted_dataset/train_label"))

2400

In [ ]:
import os
import numpy as np
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import torchvision.transforms as tt
import albumentations as albu

class CTDataset(Dataset):
    """a class to import CT scans into the framework"""

    def __init__(
        self,
        image_dir: str,
        label_dir: str,
        resize=False,
        augment=False,
        resize_size: tuple[int, int] = (256, 256),
    ) -> None:
        """_summary_

        Args:
            image_dir (str): pass the image directory.
            label_dir (str): pass the label directory.
            resize (bool, optional): if you need resize switch this
            to True. Defaults to True.
            resize_size (tuple[int, int], optional): if resize is True,
            pass output dimensions. Defaults to (128, 128).
        """
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.images = os.listdir(image_dir)
        self.labels = os.listdir(label_dir)
        self.resize = resize
        self.augment = augment
        self.resize_size = resize_size
        self.preprocess = albu.Compose([albu.HorizontalFlip(p=0.5),
                                        albu.VerticalFlip(p=0.5),

                                        ])
        if resize:
            self.resize_func = transforms.Resize(size=resize_size, antialias=True)

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int) -> torch.Tensor:
        img_path = os.path.join(self.image_dir, self.images[index])
        label_path = os.path.join(self.label_dir, self.labels[index])

        image = np.load(img_path)
        label = np.load(label_path)




        if self.augment:
            augmented  = self.preprocess(image=image)
            image = augmented['image']
        """convert images from numpy to tensor"""
        image = transforms.Compose([
           transforms.ToTensor(),
           transforms.Normalize(np.mean([0.485, 0.456, 0.406]), np.mean([0.229, 0.224, 0.225])),
        ])(image)

        label = torch.from_numpy(label)
        label = torch.squeeze(label)

        """because of the 3D structure in pytorch we should make sure that
        input dimensions are equal to 5"""
        # image = torch.unsqueeze(image, dim=1).float()
        #label = torch.unsqueeze(label, dim=1).float()

        if self.resize:
            image = self.resize_func(image)  # normalize value between 0 and 1
            # mask = torch.round(
            #     self.resize_func(mask) / 255.0
            # )  # make sure indexes are 1 and 0
        # else:
        #     image = image / 255.0  # normalize value between 0 and 1
            # mask = torch.round(mask / 255.0)  # make sure indexes are 1 and 0

        # if self.augment:
        #     image = tt.Compose([
        #         # tt.RandomCrop(32, padding=4, padding_mode='reflect'),
        #         tt.RandomHorizontalFlip(),
        #         tt.RandomVerticalFlip()
        #         # tt.RandomRotate
        #         # tt.RandomResizedCrop(256, scale=(0.5, 0.9), ratio=(1, 1)),
        #         # tt.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        #     ])(image)
            #  NOTE: change below code proportionally to
        if image.shape[0]==1:
           image=torch.cat((image,image,image),dim=0)
        elif image.shape[0]>3:
          image=image[0:3,:,:]
        return image.float(),label



In [ ]:
##test and val load
train = CTDataset("./converted_dataset/train_img","./converted_dataset/train_label", resize=True,augment=False)
test = CTDataset("./converted_dataset/test_img","./converted_dataset/test_label",resize=True,augment=False)


print(len(train))
print(len(test))


2400
600


In [ ]:
#print(np.load(".\\converted_dataset\\ct_scans\\049_0.npy"))

In [ ]:
import torch.nn.functional as F
def training_step(model, batch):
    images, labels = batch
    out = model(images)                  # Generate predictions
    loss = F.cross_entropy(out, labels) # Calculate loss
    return loss


def validation_step(self, batch):
    images, labels = batch
    out = self(images)                    # Generate predictions
    loss = F.cross_entropy(out, labels)   # Calculate loss
    acc = accuracy(out, labels)           # Calculate accuracy
    return {'val_loss': loss.detach(), 'val_acc': acc}

def test_step(self, batch):
    images, labels = batch
    out = self(images)                    # Generate predictions
    loss = F.cross_entropy(out, labels)   # Calculate loss
    acc = accuracy(out, labels)           # Calculate accuracy
    return {'test_loss': loss.detach(), 'test_acc': acc}


def validation_epoch_end(self, outputs):
    batch_losses = [x['val_loss'] for x in outputs]
    epoch_loss = torch.stack(batch_losses).mean()   # Combine losses
    batch_accs = [x['val_acc'] for x in outputs]
    epoch_acc = torch.stack(batch_accs).mean()      # Combine accuracies
    return {'val_loss': epoch_loss.item(), 'val_acc': epoch_acc.item()}
def test_epoch_end(self, outputs):
    batch_losses = [x['test_loss'] for x in outputs]
    epoch_loss = torch.stack(batch_losses).mean()   # Combine losses
    batch_accs = [x['test_acc'] for x in outputs]
    epoch_acc = torch.stack(batch_accs).mean()      # Combine accuracies
    return {'test_loss': epoch_loss.item(), 'test_acc': epoch_acc.item()}

def epoch_end(self, epoch, result):
    print("Epoch [{}], train_loss: {:.4f}, val_loss: {:.4f}, val_acc: {:.4f},test_loss: {:.4f}, test_acc: {:.4f}".format(
        epoch, result['train_loss'], result['val_loss'], result['val_acc'], result['test_loss'], result['test_acc']))

def accuracy(outputs, labels):
    _, preds = torch.max(outputs, dim=1)
    return torch.tensor(torch.sum(preds == labels).item() / len(preds))


In [ ]:
from sklearn.metrics import precision_score, recall_score,f1_score,precision_recall_curve,confusion_matrix
def cm(model,batch):
  img,label=batch
  out=model(img)
  _, preds = torch.max(out, dim=1)
  o=torch.Tensor.cpu(preds).detach().numpy()
  l=torch.Tensor.cpu(label).detach().numpy()
  return confusion_matrix(o,l)
def recall(model,batch):
  img,label=batch
  out=model(img)
  _, preds = torch.max(out, dim=1)
  o=torch.Tensor.cpu(preds).detach().numpy()
  l=torch.Tensor.cpu(label).detach().numpy()
  return recall_score(o,l)
def ps(model,batch):
  img,label=batch
  out=model(img)
  _, preds = torch.max(out, dim=1)
  o=torch.Tensor.cpu(preds).detach().numpy()
  l=torch.Tensor.cpu(label).detach().numpy()
  return precision_score(o,l)
def f1(model,batch):
  img,label=batch
  out=model(img)
  _, preds = torch.max(out, dim=1)
  o=torch.Tensor.cpu(preds).detach().numpy()
  l=torch.Tensor.cpu(label).detach().numpy()
  return f1_score(o,l)


In [ ]:
def get_default_device():
    """Pick GPU if available, else CPU"""
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

def to_device(data, device):
    """Move tensor(s) to chosen device"""
    if isinstance(data, (list,tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

class DeviceDataLoader():
    """Wrap a dataloader to move data to a device"""
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device

    def __iter__(self):
        """Yield a batch of data after moving it to device"""
        for b in self.dl:
            yield to_device(b, self.device)

    def __len__(self):
        """Number of batches"""
        return len(self.dl)

In [ ]:
device = get_default_device()
device

device(type='cuda')

In [ ]:
to_device(model, device);

In [ ]:
from torch.utils.data import Dataset, DataLoader,TensorDataset,random_split,SubsetRandomSampler, ConcatDataset
batch_size=100
test_loader = DataLoader(test, batch_size=batch_size)
test_loader = DeviceDataLoader(test_loader, device)
train_loader = DataLoader(train, batch_size=batch_size)
train_1d=test_loader = DeviceDataLoader(train_loader, device)
val_1d=test_loader

In [ ]:
@torch.no_grad()
def evaluate(model, val_loader,test_loader):
    model.eval()
    outputs1 = [validation_step(model,batch) for batch in val_loader]
    outputs2 = [test_step(model,batch) for batch in test_loader]
    cm1 = [cm(model,batch) for batch in val_loader]
    rec = [recall(model,batch) for batch in test_loader]
    dict_2=test_epoch_end(model,outputs2)
    dict_1=validation_epoch_end(model,outputs1)
    dict_1.update(dict_2)
    return dict_1,cm1,rec

def fit(epochs, lr, model, train_loader, val_loader,test_loader, opt_func=torch.optim.SGD):
    history = []
    history_recal=[]
    gamma=0.9
    for epoch in range(epochs):
        optimizer = opt_func(filter(lambda p: p.requires_grad, model.parameters()), lr)
        # Training Phase
        model.train()
        train_losses = []
        for batch_idx, (data, targets) in enumerate(train_loader):
            batch=(data,targets)
            loss = training_step(model, batch)
            train_losses.append(loss)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        # Validation phase
        result,cm2,rec = evaluate(model, val_loader,test_loader)
        result['train_loss'] = torch.stack(train_losses).mean().item()
        epoch_end(model,epoch, result)
        history.append(result)
        history_recal.append(np.asarray(rec).mean())
        print("recal:",history_recal[-1])
        print("learning rate:",lr)
        lr=lr*gamma
    return history,history_recal

In [ ]:
num_epochs = 35
opt_func = torch.optim.Adam
#opt_func = torch.optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0006, momentum=0.9)
lr = 0.001

In [ ]:
history,history_recal= fit(num_epochs, lr, model, train_1d, val_1d, test_loader,opt_func)

Epoch [0], train_loss: 1.3059, val_loss: 0.8516, val_acc: 0.5038,test_loss: 0.8516, test_acc: 0.5038
recal: 0.5019205929924818
learning rate: 0.001


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
def plot_accuracies(history):
    accuracies = [x['val_acc'] for x in history]
    plt.plot(accuracies, '-x')
    plt.xlabel('epoch')
    plt.ylabel('val_accuracy')
    plt.title('val_accuracy vs. No. of epochs');


In [ ]:
print(history)
plot_accuracies(history)

In [ ]:
plt.plot(history_recal, '-x')
plt.xlabel('epoch')
plt.ylabel('Recall')
plt.title('Recall vs. No. of epochs');

In [ ]:
accuracies = [x['test_acc'] for x in history]
plt.plot(accuracies, '-x')
plt.xlabel('epoch')
plt.ylabel('test_accuracy')
plt.title('test_accuracy vs. No. of epochs');


In [ ]:
def plot_losses(history):
    train_losses = [x.get('train_loss') for x in history]
    val_losses = [x['val_loss'] for x in history]
    plt.plot(train_losses, '-bx')
    plt.plot(val_losses, '-rx')
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.legend(['Training', 'Validation'])
    plt.title('val_loss Loss vs. No. of epochs');

In [ ]:
plot_losses(history)

In [ ]:
train_losses = [x.get('train_loss') for x in history]
test_losses = [x['test_loss'] for x in history]
plt.plot(train_losses, '-bx')
plt.plot(test_losses, '-rx')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend(['Training', 'test'])
plt.title('test_Loss vs. No. of epochs');

In [ ]:
arr=np.zeros((2,2))
for batch_idx, (data, targets) in enumerate(test_loader):
  batch=(data,targets)
  arr+=np.array(cm(model,batch))
arr

In [ ]:
test_f=[]
for batch_idx, (data, targets) in enumerate(test_loader):
  batch=(data,targets)
  test_f.append(np.array(f1(model,batch)))

print("f1 score: ",np.array(test_f).mean())

In [ ]:
test_presision=[]
for batch_idx, (data, targets) in enumerate(test_loader):
  batch=(data,targets)
  test_presision.append(np.array(ps(model,batch)))
print("precision: ",np.array(test_presision).mean())

In [ ]:
test_recall=[]
for batch_idx, (data, targets) in enumerate(test_loader):
  batch=(data,targets)
  test_recall.append(np.array(recall(model,batch)))
print("recall: ",np.array(test_recall).mean())

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
arr=np.zeros((2,2))
for batch_idx, (data, targets) in enumerate(test_loader):
  batch=(data,targets)
  arr+=np.array(cm(model,batch))
# Normalise
cmn = arr.astype('float') / arr.sum(axis=1)[:, np.newaxis]
fig, ax = plt.subplots(figsize=(2,2))
sns.heatmap(cmn, annot=True, fmt='.2f')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show(block=False)